In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/dataset/modified_dataset.csv
/kaggle/input/dataset/test.csv
/kaggle/input/cjpe/transformers/default/1/adapter_model.safetensors
/kaggle/input/cjpe/transformers/default/1/adapter_config.json
/kaggle/input/cjpe/transformers/default/1/README.md
/kaggle/input/cjpe/transformers/default/1/tokenizer.json
/kaggle/input/cjpe/transformers/default/1/tokenizer_config.json
/kaggle/input/cjpe/transformers/default/1/special_tokens_map.json
/kaggle/input/cjpe/transformers/default/1/tokenizer.model


In [2]:
!python3 -m venv /myenv
!source /myenv/bin/activate

In [3]:
%env TOKENIZERS_PARALLELISM=true
# %env WANDB_DISABLED=true
!pip install tensorflow-io --index-url=https://pypi.org/simple

!pip install --no-deps tensorflow-io
!pip install -q tensorflow bitsandbytes datasets loralib transformers accelerate peft

env: TOKENIZERS_PARALLELISM=true
ERROR: Could not find a version that satisfies the requirement bitsandbytes (from versions: none)
ERROR: No matching distribution found for bitsandbytes


In [4]:
!pip install tensorflow-io --index-url=https://pypi.org/simple


In [5]:
from huggingface_hub import notebook_login

# Login to Hugging Face
notebook_login()

#fine-tuning

In [6]:
import time
import torch
#import evaluate
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset
import random


df_train = pd.read_csv("/kaggle/input/dataset/modified_dataset.csv")
# df_train = df_train.head(50)
#df = df.drop(columns = ["Unnamed: 0"])
train_data = Dataset.from_pandas(df_train)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
     load_in_4bit=True,
     bnb_4bit_use_double_quant=True,
     bnb_4bit_quant_type="nf4",
     bnb_4bit_compute_dtype=torch.bfloat16)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto", token=".....")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():

        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

print(model)

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=64,
    # target_modules=["query_key_value"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], #specific to Llama models.
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print_trainable_parameters(model)

OUTPUT_DIR = "ckpts"
from transformers import TrainingArguments

training_arguments = TrainingArguments(
    per_device_train_batch_size=8,    
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    logging_steps=1,
    learning_rate=1e-4,
    fp16=True,
    max_grad_norm=0.3,
    num_train_epochs=5,
    evaluation_strategy="epoch",
    eval_steps=0.2,
    warmup_ratio=0.05,
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    group_by_length=True,
    output_dir=OUTPUT_DIR,
    save_safetensors=True,
    lr_scheduler_type="cosine",
    seed=42,
)
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!

from trl import SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=2060,
    tokenizer=tokenizer,
    args=training_arguments,
)

trainer.train()

peft_model_path="llama_pred_exp"

trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

PackageNotFoundError: No package metadata was found for bitsandbytes